# 06 — Les figures du dépôt, régénérées depuis les artefacts

Le notebook 05 a fermé l'histoire : l'expérience oracle, `PROTOCOL.md` §28
bout en bout. Celui-ci n'ajoute aucun résultat neuf. Il **relit les artefacts
de verdict déjà écrits sur disque** et rend chaque figure que le dépôt publie,
puis en donne la lecture — ce qu'on voit, et ce qu'on ne peut pas en conclure.

Cette lecture dépend de huit racines `results_*/`, gitignorées par convention
— `FINDINGS.md §11` : le contenu scientifique vit dans `PROTOCOL.md` et dans
les sidecars de figure, tous deux versionnés, pas dans les runs bruts. Dans
l'ordre où ce notebook les lit : `results_gate_d/`, `results_d3_oscillator/`,
`results_osc_n/`, `results_osc_delta/`, `results_ssm_delta/`,
`results_osc_decomp/`, `results_primary_factorial/`, `results_basis/`. Sur un
clone frais, sans elles, la cellule suivante échoue sur un `FileNotFoundError`
propre — attendu, pas un bug de ce notebook. Les figures restent lisibles
telles quelles dans `assets/`, committées séparément et régénérées seulement
quand les racines existent. La huitième, `results_basis/`, a fermé le
2026-08-19 (`PROTOCOL.md §28.7`) et referme la série : ses trois contrastes
P1/P2/P3 sont rendus ici, en dernière figure (§6).

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from sld.viz.ingest import arm_label, load_d3_gold, load_gate

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pd.set_option("display.width", 120)

## 1. La vérification, avant toute figure

`PROTOCOL.md §21.0` publie, par cellule, les médianes sur cinq seeds de
$e_1$, $m_\text{low}$ et $m_\text{high}$, **et** le nombre de seeds qui
franchissent chaque borne individuellement (§12.4 : pas de réducteur). On les
recalcule depuis les 40 `gate.json`.

In [ ]:
PUBLISHED = {
    ("OU", "mlp"): (0.0106, 0.505, 0.540, 5, 5),
    ("OU", "transformer"): (0.0229, 0.723, 1.172, 5, 1),
    ("OU", "mlp_k"): (0.0520, 1.148, 2.656, 1, 0),
    ("OU", "transformer_k"): (0.0492, 1.112, 2.516, 1, 0),
    ("AC", "mlp"): (0.0119, 0.966, 0.777, 3, 5),
    ("AC", "transformer"): (0.0297, 0.854, 1.934, 5, 0),
    ("AC", "mlp_k"): (0.0438, 1.690, 2.857, 0, 0),
    ("AC", "transformer_k"): (0.0436, 1.502, 2.841, 0, 0),
}

scalars, curves = load_gate(REPO / "results_gate_d")
for frame in (scalars, curves):
    frame["arm"] = [arm_label(b, s) for b, s in zip(frame.backbone, frame.structure, strict=True)]

rows = []
for (stratum, arm), sub in scalars.groupby(["stratum", "arm"]):
    e1, mlow, mhigh, n_lo, n_up = PUBLISHED[(stratum, arm)]
    rows.append(
        {
            "strate": stratum,
            "bras": arm,
            "e_1": sub.e_1.median(),
            "e_1 publié": e1,
            "m_low": sub.m_low.median(),
            "m_low publié": mlow,
            "m_high": sub.m_high.median(),
            "m_high publié": mhigh,
            "accord": (
                abs(sub.e_1.median() - e1) < 5e-5
                and abs(sub.m_low.median() - mlow) < 5e-4
                and abs(sub.m_high.median() - mhigh) < 5e-4
                and int(sub.lower_bound.sum()) == n_lo
                and int(sub.upper_bound.sum()) == n_up
            ),
        }
    )
reconciliation = pd.DataFrame(rows)
assert reconciliation["accord"].all(), "ÉCART avec §21.0 — aucune figure ne sort, l'écart est le résultat"
reconciliation

**Lecture.** Les huit cellules se reproduisent, médianes **et** comptes de
seeds. La chaîne de mesure qui alimente les figures est donc la même que
celle qui a écrit le protocole. Le `assert` ci-dessus est le verrou : si un
refactor déplaçait un chiffre, ce notebook s'arrêterait ici.

Le même contrôle vaut pour D3 (`§20.0` : 10 runs, 10/10 sur le gain,
0/10 sur les trois bornes d'exactitude).

In [ ]:
gold = load_d3_gold(REPO / "results_d3_oscillator")
print(f"D3 : {len(gold)} runs, gain {int(gold.gain_passes.sum())}/{len(gold)}")
for col in ("state_h100", "spectral", "action"):
    n = int((gold[col] < gold[f"threshold_{col.split('_')[0]}"]).sum())
    print(f"  {col:11s} médiane {gold[col].median():.4f}  seuil {gold[f'threshold_{col.split(chr(95))[0]}'].iloc[0]:g}  -> {n}/{len(gold)} passent")

## 2. Figure A — le contrôle D3 : ils apprennent, ils n'identifient pas

Quatre panneaux, quatre axes séparés, délibérément : `§19.2` grave que *la
borne spectrale ne doit jamais être rapportée comme si elle résumait les
autres*. Les mettre sur un axe commun ferait exactement cela.

Les lignes horizontales sont des **ancres analytiques** (`§20.1`), jamais des
mesures de ces runs : le gold planté à $1.9\times10^{-15}$ (l'optimum existe
dans la classe d'hypothèses), les moindres carrés sur le même split, et le
seuil pré-enregistré $10^{-3}$.

In [ ]:
from sld.viz.figA_d3_exactness import make_figA

path = make_figA(REPO / "results_d3_oscillator", REPO / "assets/figA_d3_exactness.png")
print(path)
display(Image(str(path)))

**Lecture.** Panneau 1 : les dix cellules battent les baselines triviales
d'environ $32\times$ — elles apprennent, et la porte relative est franchie
10/10. Panneaux 2 à 4 : les trois bornes d'exactitude absolue sont manquées
de **110 à 200 fois** le seuil pré-enregistré.

Ce qu'on **ne peut pas** dire : « 200 fois plus loin que les moindres
carrés ». L'écart réel aux MCO est de l'ordre de $5\times10^{6}$, et la
légende se contredirait. Ce qu'on **peut** dire, et c'est le résultat : sur un
système linéaire dont la solution exacte est dans la classe d'hypothèses *et*
récupérable en forme close, le régime d'entraînement déclaré ne l'atteint pas.

Portée, à ne jamais élargir : cet énoncé porte sur **cette** paramétrisation,
**ce** régime (rollout fermé 25 pas), **cet** optimiseur. Il ne dit rien des
priors structurés en général.

## 3. Figure B — la grille D4, lue à travers l'horizon

`§21.0` rapporte les marges de porte à un horizon unique et conclut que
`structure: on` coûte $1.54\times$ à $2.27\times$. C'est exact, et ce n'est
pas tout ce que portent les artefacts : les mêmes `gate.json` contiennent
$e_h$ pour tout $h \in \{1, \dots, 25\}$.

In [ ]:
from sld.viz.figB_horizon import make_figB

path = make_figB(REPO / "results_gate_d", REPO / "assets/figB_horizon.png")
print(path)
display(Image(str(path)))

e1 = curves[curves.h == 1].set_index(["stratum", "arm", "seed"])["e_h"]
e25 = curves[curves.h == 25].set_index(["stratum", "arm", "seed"])["e_h"]
rho = (e25 / e1).groupby(level=[0, 1]).median().rename("rho = e_25 / e_1")
rho.to_frame()

**Lecture.** Le taux de composition $\rho = e_{25}/e_1$ vaut, en médiane :

| strate | OFF | ON | rapport |
|---|---|---|---|
| OU `mlp` | 6.24 | 3.03 | **2.06×** |
| OU `transformer` | 4.26 | 2.66 | **1.60×** |
| AC `mlp` | 8.58 | 3.92 | **2.19×** |
| AC `transformer` | 3.42 | 3.72 | 0.92× |

Dans trois paires sur quatre, le bras structuré part d'une erreur à un pas
plus grande et **compose plus lentement**. C'est, mot pour mot, la question de
recherche affichée du dépôt.

**Deux confondants, ni l'un ni l'autre réparable depuis ces artefacts**, et
aucune lecture de cette figure ne doit les omettre :

1. l'asymétrie de taux de `§21.2` — chaque bras OFF a tourné à un taux choisi
   pour lui, chaque bras ON à un taux choisi pour une *autre* architecture ;
2. `structure: on` est un **composite** — la transition structurée *et* le
   lift $\psi$ *et* la projection $\pi$, qui prennent ensemble 10.3 % à
   20.4 % du budget de paramètres (`§13.7`). Les panneaux ne séparent pas les
   trois.

C'est pourquoi ce chiffre est présenté comme un **signal à mesurer
proprement**, pas comme un résultat : les campagnes de `§23` existent pour ça.

## 4. Figures C à G — rendues

Leurs racines de résultats, déclarées dans `PROTOCOL.md §23` et `§25`, existent maintenant sur disque, et les cinq figures sont rendues dans `assets/` : les cellules suivantes les régénèrent depuis leurs artefacts, comme les figures A et B ci-dessus.

| figure | racine | ce qu'elle porte |
|---|---|---|
| **C** | `results_osc_n/` | gain de structure contre taille d'échantillon, trois niveaux, 5 seeds en points individuels |
| **D** | `results_osc_delta/` | gain contre le défaut de représentabilité $\delta(c)$, abscisse **calculée en forme close** |
| **E** | `results_ssm_delta/` | le verdict `ssm` : la famille s'entraîne-t-elle sous un $\Delta_{\max}$ physiquement borné ? |
| **F** | `results_osc_n/` | $\kappa(\partial x_H / \partial u_{0:H-1})$ par bras — la seule quantité du dossier qui touche le chaînon planification |
| **G** | `results_osc_decomp/` | retire le lift $\psi$/$\pi$ du bras structuré : la pénalité vient-elle de la transition, ou du couple lift/projection ? |

La règle de lecture primaire de C, D et E est **écrite avant les chiffres** (`§23.5`) : c'est le **signe** d'une interaction et sa taille d'effet, jamais un franchissement de seuil. Si les intervalles se recouvrent partout, le verdict publié est « aucune interaction détectée à 5 seeds sur ce système », **et la figure sort quand même**. G suit une règle analogue mais distincte, déclarée séparément (`§25.3`) : une attribution entre trois issues plutôt qu'un signe.

In [ ]:
from sld.viz.figC_sample_size import make_figC, make_figG
from sld.viz.figD_misspecification import make_figD
from sld.viz.figE_ssm_delta import make_figE
from sld.viz.figF_action_conditioning import make_figF

path = make_figC(REPO / "results_osc_n", REPO / "assets/figC_sample_size.png")
print(path)
display(Image(str(path)))

**Lecture.** Trois niveaux de structure — `off`, `on-free`, `on-stable` (§23.1) — contre trois tailles d'échantillon `n ∈ {8, 32, 256}`, cinq graines chacun, 45 cellules. Une seule diverge : `on-free` à `n = 32` graine 3, annotée « 1 div » sur le graphe plutôt que recalculée en silence (§15.3). Le balayage de taux de §23.3, mené par bras, a convergé sur `lr = 1e-3` pour les trois (§25.1). L'axe vertical est `m_low` en log, la dispersion étant le résultat affiché : chaque graine est un point, la barre noire la médiane, le trait fin le range, le trait épais l'IQR — une médiane seule effacerait la largeur du bras `on-free` à `n = 8`, visible dans le graphe : la plage y couvre plus d'un facteur sept.

L'estimand pré-enregistré, écrit avant les chiffres (§23.5), est le SIGNE de l'interaction structure × log(n) : la prédiction était que le gain de `on-stable` sur `off` diminue quand `n` grandit. Le résultat mesuré est plus simple que la prédiction ne l'anticipait : la campagne A retourne une pénalité de structure à chaque taille d'échantillon (§25.0), confirmé par `FINDINGS.md §4` — la structure perd à `n = 8`, `32` et `256`.

Ce que la figure ne montre pas : cette racine n'a pas de porte (§23.4) — la médiane affiche, elle ne décide rien, et chaque graine reste visible. L'axe structure déplace un COMPOSITE, transition et lift `ψ`/`π` ensemble (§13.7) ; aucune cellule ici n'isole la transition seule, c'est exactement la question que `results_osc_decomp/` (figure G, plus bas) a été déclarée pour trancher. Portée (§25.6) : mesuré sous le régime déclaré `h_fr = 25` ; le signe de l'effet de structure est inchangé à `h_fr = 5`, sa taille ne l'est pas (§25.8).

In [ ]:
path = make_figD(REPO / "results_osc_delta", REPO / "assets/figD_misspecification.png")
print(path)
display(Image(str(path)))

**Lecture.** Quatre niveaux de `δ(c)`, trois bras, cinq graines : 60 cellules, 58 complètes, deux divergences — `on-free` graine 3 à `c = 0.10` et à `c = 2.60` (même graine que la figure C, deux racines différentes, §15.3). Deux panneaux, dans cet ordre parce que l'ordre est le résultat. À gauche, les absolus : `m_low` des trois bras contre `δ(c)` — calculé en forme close par §23.7, pas estimé — avec les deux baselines triviales sur un axe jumeau. À droite, le ratio structuré/`off`, jamais montré seul.

L'estimand pré-enregistré (§23.5) est le SIGNE de la pente du gain de structure contre `δ(c)` ; la prédiction — le gain diminue quand `δ` croît — est confirmée : le ratio de coût monte de **+9.61** par unité de `δ` pour `on-free` et **+10.52** pour `on-stable` (§26.1). Le signe tient.

Ce que le panneau de gauche ajoute, et que le ratio seul cache : les deux bras structurés ne se dégradent presque pas — `on-stable` varie ×1.24 sur tout l'axe, `on-free` ×1.20, tous deux ÉPINGLÉS près de `m_low ≈ 1` (§26.1). C'est `off` qui bouge, et dans le sens qui compte : ×3.67, en s'améliorant quand `c` croît. La hausse du ratio est donc un effet de DÉNOMINATEUR, pas une dégradation du bras structuré — le mécanisme qu'imaginait la prédiction (la structure décroche quand le prior se dégrade) n'est pas celui qui est mesuré (la structure est indifférente au prior, à un niveau de coût déjà mauvais).

Ce que la figure ne montre pas : ni gate (§23.4) ni confondant réparé — l'axe structure reste un composite transition + `ψ`/`π` (§13.7), et l'argmin du taux de §23.3 était, comme pour la figure C, au bord de la liste déclarée. `δ(c)` lui-même dépend d'une convention d'appariement des valeurs propres déclarée avant les runs (§23.7) ; sous l'appariement libre, alternatif, les valeurs changent (`c = 2.60` donnerait 0.529 au lieu de 0.706) sans changer le signe de la pente, seul objet de cette lecture.

In [ ]:
path = make_figE(REPO / "results_ssm_delta", REPO / "assets/figE_ssm_delta.png")
print(path)
display(Image(str(path)))

**Lecture.** Vingt cellules, `min_ssm`/`ssm_k` croisés avec `delta_max ∈ {10.0, 0.20}` (respectivement 500× et 4× le pas physique `dt`), cinq graines. Panneau gauche : `train_loss` contre le nombre de mises à jour, axe x en log pour loger les deux régimes. Panneau droit : la norme du gradient au premier pas d'optimiseur, par graine, en log — apparié pour que la comparaison entre `delta_max` se lise graine par graine.

Le résultat est univoque et il tient en deux colonnes (§24.0) : **0/5** de chaque famille termine à `delta_max = 10.0`, **5/5** des deux familles terminent les 8000 mises à jour à `delta_max = 0.20`. `min_ssm` à 10.0 n'a même pas de courbe — `val_loss` non fini dès l'époque 0, sur les cinq graines, avant qu'une seule ligne n'atteigne `metrics.csv`, ce que le panneau gauche annonce plutôt que de le faire deviner. `18.0` est amendé en conséquence (§24.2) : la famille `ssm` n'a pas de taux entraînable à `delta_max = 10.0` ; sous un pas latent physiquement borné, elle s'entraîne.

Le panneau droit est ce qui a corrigé la première version de cette section (§24.1) : pour `ssm_k`, les deux niveaux de `delta_max` partent du MÊME premier pas — les normes de gradient s'accordent à trois décimales, graine par graine (1.807/1.806, 2.068/2.067, …). Le clamp du gate sélectif agit dès la première mise à jour mais sa conséquence est cumulative : les trajectoires de perte ne divergent qu'à l'époque 1, de l'ordre de `1e-5`, et il faut 28 à 56 mises à jour pour que ça devienne terminal. `min_ssm`, lui, meurt d'un coup, sans pas du tout — deux mécanismes de rupture différents sous le même nom « diverge ».

Ce que la figure ne montre pas : « s'entraîne » n'est pas « apprend », et cette racine n'a pas de porte (§23.4) — pour mémoire, 10/10 cellules bornées passent `m_low < 1` mais seules 2/10 passent aussi la borne supérieure. Le retrait de la famille `ssm` de la grille compétitive (`18.0`) n'est pas annulé — ce sont les coûts et le calendrier de la phase D qui l'avaient motivé, pas seulement l'entraînabilité (§24.3). Et la figure ne dit pas POURQUOI `ssm_k` met 28 à 56 mises à jour à mourir là où `min_ssm` meurt tout de suite : le mécanisme est borné, pas identifié.

In [ ]:
path = make_figF(REPO / "results_osc_n", REPO / "assets/figF_action_conditioning.png")
print(path)
display(Image(str(path)))

**Lecture.** `κ(∂x_H/∂u)` à `H = 25`, la sensibilité de l'état final du rollout libre à la séquence d'actions qui l'a produit — un `κ` élevé signale une direction d'action que le planificateur peut à peine faire sentir dans l'état final, donc doit inverser au prix fort. Même racine que la figure C (`results_osc_n/`, 44 des 45 cellules — la cellule divergente n'a pas de checkpoint et n'est pas simulée), mais lue depuis `action_jacobian.json` plutôt que `gate.json` ; état sonde et séquence d'action viennent d'UNE seule graine fixe pour chaque bras, pour que seul le modèle diffère. Chaque valeur a été recroisée contre des différences finies centrées avant d'être écrite, désaccord relatif maximal sur les 44 runs : `4.7e-09`, bien sous le `1e-8` où l'audit refuserait.

Aucune section de `PROTOCOL.md` ne porte de règle de lecture ni de verdict pré-enregistré pour cette figure — contrairement à C, D et E, `κ` est publié comme diagnostic, pas comme estimand. Lue directement : `on-free` a la pire conditionnalité de loin, médiane autour de 105-110 aux trois `n`, avec des points jusqu'à 350 ; `off` et `on-stable` restent dans le même ordre de grandeur l'un que l'autre, entre 15 et 60.

Ce que ça ne dit pas : `κ` mesure le composite structure + `ψ` + `π` (§13.7), pas la transition seule — la même limite que la figure C — et cette racine n'a pas de porte non plus (§23.4).

In [ ]:
path = make_figG(REPO / "results_osc_decomp", REPO / "assets/figG_decomposition.png")
print(path)
display(Image(str(path)))

**Lecture.** `results_osc_decomp/` répond à une question que la figure C ne peut pas trancher seule : la pénalité de structure vient-elle de la transition, ou du couple lift/projection `ψ`/`π` qui l'accompagne ? Un troisième bras, `on-stable-nolift`, applique la transition structurée directement sur `z`, avec `ψ = π =` identité — 30 cellules, `off` / `on-stable` / `on-stable-nolift` × `n ∈ {8, 256}` × cinq graines, au même `h_fr = 25` que la figure C, fixé par §25.3 avant que la racine 1 de l'amendement 16 ne réponde.

La règle de lecture, écrite avant les chiffres (§25.3), prédisait trois issues : `nolift` proche de `off` (la pénalité vient de `ψ`/`π`), proche de `on-stable` (elle vient de la transition), ou entre les deux (attribution partielle). Le résultat mesuré est une quatrième chose, extérieure à la partition prévue : `nolift` se place AU-DESSUS des deux autres bras, **3.1×** pire que `on-stable` à `n = 8`, **13.1×** à `n = 256` (§26.0, `FINDINGS.md §5`).

La raison est un calcul en forme close, pas une panne d'optimiseur (§26.0) : dans la base identité, le générateur vrai `A` est à distance de Frobenius **1.0654** de la classe rotation-échelle par blocs — **36 %** de `‖A‖_F = 2.9614` — et à **1.97e-15** de cette même classe dans sa propre base de Jordan réelle. `on-stable` peut apprendre ce changement de base par `ψ`/`π` ; `nolift`, par construction, ne le peut pas et reste coincé dans la base identité. `ψ` et `π` ne sont donc pas un surcoût au-dessus de la transition structurée : ils sont le changement de base que la structure exige.

Ce que la figure ne montre pas : deux confondants déclarés avant les runs — `nolift` porte 15.7 % de paramètres en moins par construction (§23.2 gèle le dorsal plutôt que d'ajuster le budget), et elle hérite du taux de `on-stable` puisque §23.5(2) interdit un second balayage. Ni gate, ni seuil, sur cette racine non plus (§23.4).

## 5. Figures 1 à 3 — la racine factorielle primaire

Chaque cellule du cycle oscillateur (`results_sweep_osc/`, `results_osc_n/`, `results_osc_delta/`, `results_osc_decomp/`, `results_osc_hfr/`) a tourné sur le dorsal `mlp` ; `results_ssm_delta/` ajoute `min_ssm`/`ssm_k` sur sa propre question à un facteur, jamais en comparaison (`PROTOCOL.md §27.0`). Le négatif que les figures C, D, E et G documentent n'était donc mesuré que sur UN dorsal, quand le design du dépôt en croise trois. `results_primary_factorial/` referme ce trou, en racine séparée et autosuffisante (§27.1) : 30 cellules, trois dorsaux (`mlp`, `transformer`, `ssm`) × structure (`off`/`on`) × cinq graines.

Trois figures, engagées avant les données (§27.4) : la courbe `e_h` (Fig. 1), le VPT brut par graine (Fig. 2), et le contraste apparié ON−OFF par dorsal (Fig. 3).

In [ ]:
from sld.viz.fig1_rollout import make_fig1
from sld.viz.fig2_vpt import make_fig2
from sld.viz.fig3_factorial_effects import make_fig3

result1 = make_fig1(REPO / "results_primary_factorial", "primary_factorial", REPO / "assets/fig1_rollout.png")
path = result1["image_path"]
print(path)
display(Image(str(path)))

**Lecture.** `e_h` en coordonnées d'état vraies, moyenne ± 1 SE sur cinq graines, six cellules — trois dorsaux (`mlp`, `transformer`, `ssm`) × structure (`off`/`on`), `H = 25` (§27.1). Une cellule NON ÉLIGIBLE (moins de 5/5 graines terminales et `ok`, §27.3) n'est jamais dessinée comme courbe — elle est nommée dans la légende à la place. Ici, quatre des six le sont : `mlp_k`, `transformer_k`, `min_ssm` et `ssm_k` portent toutes le verdict `does_not_learn` sur leurs cinq graines (§27.8) ; seules `mlp` et `transformer` restent à tracer.

La mesure (§27.8) : `e_H` moyen à `H = 25`, plus bas est mieux — **0.0702 ± 0.0040** pour `mlp`, **0.1197 ± 0.0106** pour `transformer`. `transformer` est pire, et c'est la seule relation confirmatoire que cette racine tranche sous la règle d'éligibilité pré-enregistrée : une relation `contradicted`, huit `indeterminate`, une `not_tested` (§27.8.1).

`does_not_learn` recouvre deux échecs différents et la figure ne les distingue pas — le détail vit dans §27.8.2 : `mlp_k` passe la borne inférieure neutre-au-bras sur 4/5 graines et échoue seulement la borne supérieure (`tau_1`, la même que 21.4 avait déjà notée) ; `ssm_k` échoue la borne inférieure sur 5/5, il ne bat même pas les baselines triviales.

Ce que la figure ne montre pas : les trois dorsaux n'ont PAS le même taux d'apprentissage (`mlp`/`transformer` à `1e-3`, `ssm` à `3e-4`, §27.2) — l'ordre ENTRE dorsaux hérite de ce confondant, nommé et non réparé. Seul le contraste apparié PAR dorsal (Fig. 3, plus bas) en est libre. Portée (§27.5) : `CoupledOscillators`, `OURegime`, `n = 32`, `h_fr = 25` — aucune extension au pendule.

In [ ]:
result2 = make_fig2(REPO / "results_primary_factorial", "primary_factorial", REPO / "assets/fig2_vpt.png")
path = result2["image_path"]
print(path)
display(Image(str(path)))

**Lecture.** Le **VPT** (Valid Prediction Time) : le premier pas `h` où l'erreur d'état normalisée franchit le seuil `τ` sur le rollout libre en coordonnées d'état vraies — plus grand vaut mieux (Vlachas et al. 2020, arXiv:1910.05266 ; §27.3, `τ = 0.5` ici). Par run, la valeur est la médiane sur les épisodes d'évaluation ; par cellule, la moyenne sur les cinq graines. Un run dont l'erreur ne franchit jamais `τ` avant `H = 25` est `censored` — jamais remplacé par 0 ni par `H+1`, dessiné à un plafond étiqueté qui n'est qu'un EMPLACEMENT, pas une valeur (§27.3).

Chaque graine de chaque cellule est dessinée, toujours — contrairement à la figure 1, aucune cellule n'est cachée ici (§27.4). Le résultat se lit directement dans le graphe : sur les 30 graines de cette racine, **29 sont censurées**, la seule exception étant `ssm_k`, une graine, `VPT = 10`.

C'est pour ça que le VPT n'ordonne rien ici (§27.7) : sur les 162 runs déjà notés des six racines précédentes, 1179 des 1346 épisodes d'évaluation — **87.6 %** — sont censurés, et 30 runs seulement atteignent `e_h ≥ 0.5` avant `H = 25`. Le VPT est censuré PAR CONSTRUCTION sur ce système : une lecture VPT-primaire renverrait `indeterminate` sur la quasi-totalité des neuf relations, quel que soit le modèle. `e_H` à `H = 25` (Fig. 1/3) est la statistique d'ordre à la place — le VPT reste publié pour la trace pré-enregistrée, jamais retiré, mais il ne décide rien.

In [ ]:
result3 = make_fig3(REPO / "results_primary_factorial", "primary_factorial", REPO / "assets/fig3_factorial_effects.png")
path = result3["image_path"]
print(path)
display(Image(str(path)))

**Lecture.** Le contraste apparié `on − off` par dorsal, même graine, même jeu de données, même taux, même budget de part et d'autre (§27.2) — la SEULE comparaison de cette racine libre du confondant de taux entre dorsaux, et le second panneau des trois figures que `README.md` affiche en tête. Un contraste négatif dirait que la structure aide ; ici, positif partout, la structure coûte (§27.8.3, `FINDINGS.md §4`) :

| dorsal | contraste `on − off` | SE | graines positives |
|---|---|---|---|
| `mlp` | **+0.1451** | 0.0351 | **5/5** |
| `transformer` | **+0.2484** | 0.0670 | **5/5** |
| `ssm` | +0.2836 | 0.2379 | 3/5 |

Les deux premières colonnes portent le résultat : chaque graine, sans exception, place le bras structuré derrière. La colonne `ssm` est publiée et ne compte pas comme une troisième confirmation (§27.9) — sa moyenne est fixée par une seule graine (`+1.2141`), la médiane des cinq vaut `+0.0016`, et les deux bras qu'elle compare échouent déjà la borne inférieure neutre-au-bras (`ssm_k` 5/5, `min_ssm` 3/5, §27.8.2) : c'est un contraste entre deux bras dont ni l'un ni l'autre n'a clairement quitté les baselines triviales.

Ce que la figure établit (§27.8.3) : le négatif mesuré sur `mlp` seul par les six racines précédentes n'était pas une propriété du dorsal `mlp` — il généralise à `transformer`. Ce qu'elle ne montre pas : rien sur `ssm` à cette taille d'échantillon (`n = 32`), et rien qui compare les TROIS contrastes entre eux — cette comparaison-là hérite du confondant de taux de §27.2 et n'est pas ce que cette figure revendique.

## 6. Figure H — la racine de base, oracle contre appris

`results_basis/` (`PROTOCOL.md §28`) retire une seule chose au bras structuré : l'obligation de découvrir la vraie base par descente de gradient. Un backbone (`mlp`), trois bras, cinq graines chacun, quinze `gate.json` : `off` (la colonne `mlp` de §27), `on-learned` (structuré, `ψ`/`π` découverts avec l'opérateur — exactement le `mlp_k` de §27), `on-oracle` (structuré, `ψ`/`π` GELÉS à la vraie base, zéro paramètre de lift entraînable, construits depuis le générateur vrai qu'aucun autre bras du dépôt n'a le droit de voir, §28.2).

C'est la dernière racine du dépôt (§28.5) : une troisième levée du plafond n'est plus possible sur la seule signature de l'auteur, elle exige désormais un objecteur nommé qui n'est pas lui.

In [ ]:
from sld.viz.figH_basis_oracle import make_figH

path = make_figH(REPO / "results_basis", REPO / "assets/figH_basis_oracle.png")
print(path)
display(Image(str(path)))

**Lecture.** `e_H` à `H = 25`, moyenne sur cinq graines par bras (panneau gauche, lignes fines : même graine à travers les trois bras) et les trois contrastes appariés au même seed (panneau droit), règle de lecture de `§28.3`. Statut de contrôle positif, publié à côté des chiffres et **jamais utilisé pour l'éligibilité** (`§28.3`) : `off` **5/5** `learns`, `on-learned` **0/5**, `on-oracle` **5/5**. Moyennes sur cinq graines : `off` **0.0702**, `on-learned` **0.2153**, `on-oracle` **0.0851**.

P1 (`on-learned − off`) est le **contrôle** de cette racine, pas une découverte : il réplique la colonne `mlp` de `§27.8.3` à la précision affichée — **+0.1451 ± 0.0351**, 5+/0−, `supported`. La réplication à la décimale est un **déterminisme** — même graine, même `data_hash`, même schedule, réentraîné de zéro dans cette racine sous `§14.6` — pas un tirage indépendant ; un échec ici aurait rendu P2 et P3 illisibles (`§28.4`).

P2 (`on-oracle − on-learned`) est `supported`, **0+/5−** : **−0.1302 ± 0.0340**. Donner la vraie base retire la majorité de la pénalité sur presque chaque graine et rend au bras structuré le gate de contrôle positif qu'il ratait 5/5 en base apprise.

P3 (`on-oracle − off`) est `indeterminate`, **3+/2−** : **+0.0149 ± 0.0189**, l'intervalle à ±1 SE chevauche zéro. Donner la base ne suffit pas à mettre la classe structurée devant la classe non structurée.

`on-oracle` est un **plafond**, pas un concurrent loyal (`§28.2`) : sa base lui est donnée — `ψ`/`π` gelés, zéro paramètre entraînable — ce qu'aucun autre bras du dépôt n'a le droit de voir. Sous P2 `supported` et P3 non `supported`, l'énoncé que `§28.4` fixait avant les quinze cellules, retenu par `§28.7` : *« the coupling is a contributing mechanism; it is not sufficient to reverse the sign »* — jamais « la structure marche quand on fixe la base ».

Ce que la figure ne montre pas : aucun test sur `transformer` ou sur `ssm`, aucun sur `DampedPendulum` (`§28.5` — `M4` n'est pas franchi) ; aucun seuil, taux ou budget ne bouge. Et `§27.8.1`, `§27.8.3`, `§27.9` tiennent exactement comme écrits, quel que soit le verdict de cette racine (`§28.4`).

## 7. Conclusion

Ce notebook a relu huit racines de résultats (`results_gate_d/`, `results_d3_oscillator/`, `results_osc_n/`, `results_osc_delta/`, `results_ssm_delta/`, `results_osc_decomp/`, `results_primary_factorial/`, `results_basis/`) à travers onze figures, sans jamais recalculer une seule cellule. La question qu'elles répondent ensemble est celle du dépôt entier : sur `CoupledOscillators`, où le prior rotation-échelle est exactement correct par construction, la structure explicite paie-t-elle ? `FINDINGS.md` (Summary) la pose sur 194 cellules et sept racines pour le négatif seul, plus une huitième, 15 cellules, pour l'oracle qui referme la série.

**Ce qui est établi.** Le contraste apparié — la seule mesure libre de confondant, off et on d'un même dorsal partageant graine, jeu de données, taux et budget — est positif sur les deux dorsaux où il est fiable : `mlp` (`+0.1451 ± 0.0351`, 5/5 graines, Fig. 3) et `transformer` (`+0.2484 ± 0.0670`, 5/5 graines, Fig. 3). La structure coûte, et ce négatif tient à `n = 8`, `32` et `256` (Fig. C), à `δ(c)` de `0.000` à `1.029` (Fig. D), et sur les deux horizons d'entraînement mesurés (§25.8). Retirer le lift `ψ`/`π` ne réduit pas la pénalité, il l'aggrave — 3.1× à 13.1× pire, hors de la partition prédite (Fig. G) — parce que `ψ`/`π` ne sont pas un surcoût sur la transition structurée : ils sont le changement de base que la classe rotation-échelle exige, et ce changement de base est loin d'être l'identité (36 % de `‖A‖_F`, §26.0).

**La leçon de la base**, mesurée en dernier (§28.7, citée dans l'introduction) : donner ce changement de base tout fait à la structure efface l'essentiel de la pénalité, sur les cinq graines (`on-oracle − on-learned = -0.1302 ± 0.0340`) et lui rend la porte de contrôle positif qu'elle ratait 5/5 en base apprise ; il ne suffit toutefois pas à la faire passer devant le dorsal non structuré (`on-oracle − off = +0.0149 ± 0.0189`, indéterminé). L'énoncé que §28.4 fixait avant les quinze cellules, retenu par §28.7 : *« the coupling is a contributing mechanism; it is not sufficient to reverse the sign »* — le couplage base + opérateur ne contribue pas seulement, il ne suffit pas à renverser le signe.

**Pointeurs `sld`.** `sld.viz.ingest` (`load_gate`, `load_d3_gold`, `arm_label`) porte la relecture des cellules 1, 3 et 5 ; `sld.viz.results` (`build_primary_results_table`, `aggregate_rollout_curves`, `summarize_vpt`, `paired_structure_contrasts`) porte les figures 1 à 3 ; `sld.viz.style` porte la palette et les conventions partagées par les onze figures (`FAMILY_COLORS`, `STRUCTURE_COLORS`, `save_fig`, `write_sidecar_csv`) ; chaque `sld.viz.figX_*` porte sa propre figure et rien d'autre.

**La limite.** Rien ici n'atteint le pendule amorti — `M4` n'est pas franchi, sept des huit cellules de contrôle positif y échouent (§21.4), et aucun énoncé de ce notebook ne s'étend à ce système. La colonne `ssm` du contraste apparié n'est pas une troisième confirmation à `n = 32` (§27.9), bien qu'elle s'entraîne sous un pas latent physiquement borné à une autre taille d'échantillon (Fig. E). Et le bras `on-oracle` est un PLAFOND, pas un concurrent loyal — sa base lui est donnée, ce qu'aucun autre bras n'a le droit de voir (§28.2) ; ce que la Fig. 3 mesurerait pour lui reste hors de portée d'un entraînement réel.

**Le pont.** `notebooks/04_ce_que_la_grille_a_dit.ipynb` §6.1 tient le ledger d'honnêteté qui recense ce contraste et ce que la racine de base en a mesuré. Ce notebook en est la couche visuelle : la lecture complète, section par section, amendement par amendement, vit dans `PROTOCOL.md` — 28 sections, dix-huit amendements datés, chacun commité avant le run qu'il autorise (`REGLES.md`, R4). `FINDINGS.md` en est le résumé en six pages.